# Kapitola 6: Předvídání (Myslení krok za krokem)

- [Lekce](#lesson)
- [Cvičení](#exercises)
- [Příkladové hřiště](#example-playground)

## Nastavení

Spusťte následující buňku nastavení pro načtení vašeho API klíče a vytvoření pomocné funkce `get_completion`.

In [ ]:
%pip install anthropic --quiet

# Importujte modul hints z balíčku utils
import os
import sys
module_path = ".."
sys.path.append(os.path.abspath(module_path))
from utils import hints

# Importujte vestavěnou knihovnu regulárních výrazů v Pythonu
import re
from anthropic import AnthropicBedrock

%store -r MODEL_NAME
%store -r AWS_REGION

client = AnthropicBedrock(aws_region=AWS_REGION)

def get_completion(prompt, system='', prefill=''):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": prompt},
          {"role": "assistant", "content": prefill}
        ],
        system=system
    )
    return message.content[0].text

---

## Lekce

Pokud by vás někdo probudil a okamžitě začal klást několik složitých otázek, na které byste museli hned odpovědět, jak byste si vedli? Pravděpodobně ne tak dobře, jako kdybyste měli čas **nejprve promyslet svou odpověď**.

Hádejte co? Claude je na tom stejně.

**Dát Claudovi čas přemýšlet krok za krokem někdy činí Clauda přesnějším**, zejména u složitých úkolů. Nicméně, **přemýšlení se počítá jen tehdy, když je nahlas**. Nemůžete požádat Clauda, aby přemýšlel, ale výstupem byla pouze odpověď - v tomto případě k žádnému přemýšlení ve skutečnosti nedošlo.

### Příklady

V následujícím promptu je pro lidského čtenáře jasné, že druhá věta odporuje první. Ale **Claude bere slovo "unrelated" příliš doslovně**.

In [ ]:
```python
# Prompt
PROMPT = """Is this movie review sentiment positive or negative?

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since the year 1900."""

# Vytisknout odpověď od Claude
print(get_completion(PROMPT))
```

Abychom zlepšili odpověď od Claude, pojďme **dovolit Claudovi nejprve promyslet věci, než odpoví**. Toho dosáhneme tím, že doslova vyjmenujeme kroky, které by měl Claude podniknout, aby zpracoval a promyslel svůj úkol. Spolu s trochou role promptingu to umožňuje Claudovi lépe pochopit recenzi.

In [ ]:
```python
# Systémový prompt
SYSTEM_PROMPT = "You are a savvy reader of movie reviews."

# Prompt
PROMPT = """Is this review sentiment positive or negative? First, write the best arguments for each side in <positive-argument> and <negative-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since 1900."""

# Vytisknout odpověď od Claude
print(get_completion(PROMPT, SYSTEM_PROMPT))
```

**Claude je někdy citlivý na pořadí**. Tento příklad je na hranici Claudeovy schopnosti porozumět nuancovanému textu, a když vyměníme pořadí argumentů z předchozího příkladu tak, že negativní je první a pozitivní druhý, změní to Claudeovo celkové hodnocení na pozitivní.

Ve většině situací (ale ne ve všech, což je poněkud matoucí), **Claude má větší pravděpodobnost, že zvolí druhou z dvou možností**, možná proto, že v jeho tréninkových datech z webu byly druhé možnosti častěji správné.

In [ ]:
```python
# Prompt
PROMPT = """Is this review sentiment negative or positive? First write the best arguments for each side in <negative-argument> and <positive-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. Unrelatedly, I have been living under a rock since 1900."""

# Vytisknout odpověď od Claude
print(get_completion(PROMPT))
```

**Nechat Clauda přemýšlet může změnit jeho odpověď z nesprávné na správnou**. Je to tak jednoduché v mnoha případech, kdy Claude dělá chyby!

Pojďme si projít příklad, kde je Claudeova odpověď nesprávná, abychom viděli, jak může požádání Clauda o přemýšlení tuto chybu napravit.

In [ ]:
```python
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956."

# Vytisknout odpověď od Claude
print(get_completion(PROMPT))
```

Pojďme to opravit tím, že požádáme Claude, aby přemýšlel krok za krokem, tentokrát v tagách `<brainstorm>`.

In [ ]:
```python
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956. First brainstorm about some actors and their birth years in <brainstorm> tags, then give your answer."

# Vytisknout odpověď od Claude
print(get_completion(PROMPT))
```

Pokud byste chtěli experimentovat s prompty lekce, aniž byste měnili jakýkoli obsah výše, přejděte až na konec notebooku lekce a navštivte [**Example Playground**](#example-playground).

---

## Cvičení
- [Cvičení 6.1 - Klasifikace e-mailů](#exercise-61---classifying-emails)
- [Cvičení 6.2 - Formátování klasifikace e-mailů](#exercise-62---email-classification-formatting)

### Cvičení 6.1 - Klasifikace e-mailů
V tomto cvičení budeme instruovat Claude, aby třídil e-maily do následujících kategorií:
- (A) Dotaz před prodejem
- (B) Rozbitá nebo vadná položka
- (C) Dotaz na fakturaci
- (D) Jiné (prosím vysvětlete)

Pro první část cvičení změňte `PROMPT`, aby **Claude správně vypsal klasifikaci a POUZE klasifikaci**. Vaše odpověď musí **zahrnovat písmeno (A - D) správné volby, včetně závorek, a také název kategorie**.

Podívejte se na komentáře vedle každého e-mailu v seznamu `EMAILS`, abyste věděli, do které kategorie by měl být e-mail zařazen.

In [ ]:
```python
# Šablona promptu s místem pro proměnnou obsah
PROMPT = """Please classify this email as either green or blue: {email}"""

# Předvyplnění pro Claudeovu odpověď, pokud existuje
PREFILL = ""

# Obsah proměnné uložený jako seznam
EMAILS = [
    "Hi -- My Mixmaster4000 is producing a strange noise when I operate it. It also smells a bit smoky and plasticky, like burning electronics.  I need a replacement.", # (B) Rozbitá nebo vadná položka
    "Can I use my Mixmaster 4000 to mix paint, or is it only meant for mixing food?", # (A) Dotaz před prodejem NEBO (D) Jiné (prosím vysvětlete)
    "I HAVE BEEN WAITING 4 MONTHS FOR MY MONTHLY CHARGES TO END AFTER CANCELLING!!  WTF IS GOING ON???", # (C) Dotaz na fakturaci
    "How did I get here I am not good with computer.  Halp." # (D) Jiné (prosím vysvětlete)
]

# Správné kategorizace uložené jako seznam seznamů pro možnost více správných kategorizací na email
ANSWERS = [
    ["B"],
    ["A","D"],
    ["C"],
    ["D"]
]

# Slovník řetězcových hodnot pro každou kategorii, který se použije pro hodnocení pomocí regexu
REGEX_CATEGORIES = {
    "A": "A\) P",
    "B": "B\) B",
    "C": "C\) B",
    "D": "D\) O"
}

# Iterace přes seznam emailů
for i,email in enumerate(EMAILS):
    
    # Nahrazení textu emailu do proměnné pro místo emailu
    formatted_prompt = PROMPT.format(email=email)
   
    # Získání Claudeovy odpovědi
    response = get_completion(formatted_prompt, prefill=PREFILL)

    # Hodnocení Claudeovy odpovědi
    grade = any([bool(re.search(REGEX_CATEGORIES[ans], response)) for ans in ANSWERS[i]])
    
    # Tisk Claudeovy odpovědi
    print("--------------------------- Full prompt with variable substutions ---------------------------")
    print("USER TURN")
    print(formatted_prompt)
    print("\nASSISTANT TURN")
    print(PREFILL)
    print("\n------------------------------------- Claude's response -------------------------------------")
    print(response)
    print("\n------------------------------------------ GRADING ------------------------------------------")
    print("This exercise has been correctly solved:", grade, "\n\n\n\n\n\n")
```

❓ Pokud chcete nápovědu, spusťte buňku níže!

In [ ]:
print(hints.exercise_6_1_hint)

Stále máte potíže? Spusťte buňku níže pro příklad řešení.

In [ ]:
print(hints.exercise_6_1_solution)

### Cvičení 6.2 - Formátování klasifikace e-mailů
V tomto cvičení budeme upravovat výstup výše uvedeného promptu tak, aby poskytoval odpověď formátovanou přesně tak, jak chceme.

Použijte svou oblíbenou techniku formátování výstupu, aby Claude obalil POUZE písmeno správné klasifikace do tagů `<answer></answer>`. Například odpověď na první e-mail by měla obsahovat přesný řetězec `<answer>B</answer>`.

Pokud zapomenete, která písmenková kategorie je správná pro každý e-mail, podívejte se na komentáře vedle každého e-mailu v seznamu `EMAILS`.

In [ ]:
```python
# Šablona promptu s místem pro proměnnou obsah
PROMPT = """Please classify this email as either green or blue: {email}"""

# Předvyplnění pro Claudeovu odpověď, pokud existuje
PREFILL = ""

# Obsah proměnné uložený jako seznam
EMAILS = [
    "Hi -- My Mixmaster4000 is producing a strange noise when I operate it. It also smells a bit smoky and plasticky, like burning electronics.  I need a replacement.", # (B) Rozbitý nebo vadný předmět
    "Can I use my Mixmaster 4000 to mix paint, or is it only meant for mixing food?", # (A) Dotaz před prodejem NEBO (D) Jiné (prosím vysvětlete)
    "I HAVE BEEN WAITING 4 MONTHS FOR MY MONTHLY CHARGES TO END AFTER CANCELLING!!  WTF IS GOING ON???", # (C) Dotaz na fakturaci
    "How did I get here I am not good with computer.  Halp." # (D) Jiné (prosím vysvětlete)
]

# Správné kategorizace uložené jako seznam seznamů pro možnost více správných kategorizací na email
ANSWERS = [
    ["B"],
    ["A","D"],
    ["C"],
    ["D"]
]

# Slovník řetězcových hodnot pro každou kategorii, který se použije pro hodnocení pomocí regexu
REGEX_CATEGORIES = {
    "A": "<answer>A</answer>",
    "B": "<answer>B</answer>",
    "C": "<answer>C</answer>",
    "D": "<answer>D</answer>"
}

# Iterace přes seznam emailů
for i,email in enumerate(EMAILS):
    
    # Nahrazení textu emailu do proměnné placeholderu emailu
    formatted_prompt = PROMPT.format(email=email)
   
    # Získání Claudeovy odpovědi
    response = get_completion(formatted_prompt, prefill=PREFILL)

    # Hodnocení Claudeovy odpovědi
    grade = any([bool(re.search(REGEX_CATEGORIES[ans], response)) for ans in ANSWERS[i]])
    
    # Tisk Claudeovy odpovědi
    print("--------------------------- Full prompt with variable substutions ---------------------------")
    print("USER TURN")
    print(formatted_prompt)
    print("\nASSISTANT TURN")
    print(PREFILL)
    print("\n------------------------------------- Claude's response -------------------------------------")
    print(response)
    print("\n------------------------------------------ GRADING ------------------------------------------")
    print("This exercise has been correctly solved:", grade, "\n\n\n\n\n\n")
```

❓ Pokud chcete nápovědu, spusťte buňku níže!

In [ ]:
print(hints.exercise_6_2_hint)

### Gratulujeme!

Pokud jste vyřešili všechny úkoly až do tohoto bodu, jste připraveni přejít k další kapitole. Šťastné promptování!

---

## Příklad hřiště

Toto je prostor, kde můžete volně experimentovat s příklady promptů uvedenými v této lekci a upravovat prompty, abyste viděli, jak to může ovlivnit odpovědi Claude.

In [ ]:
```python
# Prompt
PROMPT = """Is this movie review sentiment positive or negative?

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since the year 1900."""

# Vytisknout odpověď od Claude
print(get_completion(PROMPT))
```

In [ ]:
```python
# Systémový prompt
SYSTEM_PROMPT = "You are a savvy reader of movie reviews."

# Prompt
PROMPT = """Je sentiment této recenze pozitivní nebo negativní? Nejprve napište nejlepší argumenty pro každou stranu v XML značkách <positive-argument> a <negative-argument>, poté odpovězte.

Tento film mě ohromil svou svěžestí a originalitou. V naprosto nesouvisejících zprávách žiji pod kamenem od roku 1900."""

# Vytisknout odpověď od Claude
print(get_completion(PROMPT, SYSTEM_PROMPT))
```

In [ ]:
```python
# Prompt
PROMPT = """Is this review sentiment negative or positive? First write the best arguments for each side in <negative-argument> and <positive-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. Unrelatedly, I have been living under a rock since 1900."""

# Vytisknout odpověď od Claude
print(get_completion(PROMPT))
```

In [ ]:
```python
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956."

# Vytisknout odpověď od Claude
print(get_completion(PROMPT))
```

In [ ]:
```python
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956. First brainstorm about some actors and their birth years in <brainstorm> tags, then give your answer."

# Vytisknout odpověď od Claude
print(get_completion(PROMPT))
```